# 01 — Data Pipeline

Loads every raw series used in v1, checks coverage and shows what each block of data is.

* Raw data is rebuilt by `scripts/run_all.sh` plus manual downloads — see `data/metadata/README.md`.
* Validation rules and known issues: `data/metadata/validation_report.md`.

| Block | Role |
|---|---|
| 10 ETFs | investable assets (EUR base: equity & real assets unhedged, bonds & credit hedged) |
| UST / ECB / JGB curves | synthetic bond returns, rate shocks, curve factors |
| Overnight rates (USD, EUR, JPY) | FX hedge carry, cash return, Sharpe ratio |
| FX | conversion of USD/JPY assets into EUR |
| Moody's Baa−Aaa, VIX | credit and volatility regime indicators, stress analysis |

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src import data_loader as dl

pd.set_option("display.float_format", "{:.3f}".format)
plt.rcParams["figure.figsize"] = (11, 4)

## 1. Investable ETFs

Adjusted close includes dividends, so it approximates a total-return index. HYG (2007-04-11) is the binding start date.

In [ ]:
px = dl.load_etf_prices()
coverage = pd.DataFrame({"start": px.apply(lambda s: s.first_valid_index().date()),
                         "end": px.apply(lambda s: s.last_valid_index().date()),
                         "n": px.count()})
common_start = px.dropna().index[0]
print("Common start:", common_start.date())
coverage

In [ ]:
(px.loc[common_start:] / px.loc[common_start:].iloc[0]).plot(logy=True, title="ETF total-return proxies (USD, rebased, log scale)")
plt.show()

## 2. Government yield curves

Par yields in percent. The ECB `ALL − AAA` spread measures the sovereign risk premium of non-AAA euro issuers.

In [ ]:
ust, ecb_aaa, ecb_all, jgb = dl.load_ust_curve(), dl.load_ecb_curve("AAA"), dl.load_ecb_curve("ALL"), dl.load_jgb_curve()
ten_y = pd.DataFrame({"UST 10Y": ust["10Y"], "EUR AAA 10Y": ecb_aaa["10Y"], "EUR ALL 10Y": ecb_all["10Y"], "JGB 10Y": jgb["10Y"]}).loc["2004":]
ax = ten_y.plot(title="10Y government yields (%)"); ax.axhline(0, color="k", lw=0.5); plt.show()
(ecb_all["10Y"] - ecb_aaa["10Y"]).mul(100).plot(title="Euro sovereign spread: ALL − AAA 10Y (bp)"); plt.show()

## 3. Overnight rates

The rate differential drives the cost (or gain) of hedging foreign bonds back into EUR: hedged return ≈ local return + (r_EUR − r_foreign).

In [ ]:
sr = dl.load_short_rates().loc["1999":]
sr.plot(title="Overnight rates (%): USD EFFR, EUR EONIA→€STR, JPY call"); plt.show()
(sr["EUR"] - sr["USD"]).plot(title="Annual carry of hedging USD assets into EUR (r_EUR − r_USD, %)"); plt.axhline(0, color="k", lw=0.5); plt.show()

## 4. FX

EURUSD converts USD-listed ETFs into EUR; EURJPY converts JGB returns.

In [ ]:
fx = dl.load_fx().loc["1999":]
fx[["EURUSD"]].plot(title="EURUSD (USD per EUR)"); plt.show()
fx[["EURJPY"]].plot(title="EURJPY (JPY per EUR)"); plt.show()

## 5. Credit and volatility indicators

In [ ]:
cr, vix = dl.load_credit(), dl.load_vix()
fig, ax = plt.subplots()
cr["BAA_AAA"].loc["2000":].plot(ax=ax, label="Baa − Aaa (%)")
ax2 = ax.twinx(); vix.loc["2000":].plot(ax=ax2, color="grey", alpha=0.5, label="VIX")
ax.set_title("Credit spread vs VIX"); ax.legend(loc="upper left"); ax2.legend(loc="upper right"); plt.show()

## 6. Calendar alignment

US, euro-area and Japanese trading days differ. A global inner join of daily data would drop ~7% of days, so covariances are estimated on weekly data.

In [ ]:
cal = {"US": px["SPY"].dropna().index, "EUR": ecb_aaa["10Y"].dropna().index, "JP": jgb["10Y"].dropna().index}
cal = {k: v[v >= common_start] for k, v in cal.items()}
common = cal["US"].intersection(cal["EUR"]).intersection(cal["JP"])
pd.Series({**{k: len(v) for k, v in cal.items()}, "common": len(common)}, name="days since common start")

## 7. Stress periods

In [ ]:
dl.load_stress_periods()